In [1]:
import numpy as np
import pandas as pd

from DataSplit import KFoldEvaluation
from ExtremeLearningMachine import ExtremeLearningMachine
from EvaluationMatrix import EvaluationMatrix


In [2]:
filePath = '../../Dataset/UCI_Gallstone_Dataset.csv'
df = pd.read_csv(filePath)

targetCol = ['Gallstone Status']
X = df.drop(targetCol, axis=1)
y = df[targetCol]

In [3]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [4]:
def cross_validation(X,y,activation_function,hidden_size=None,random_state=42, test_size=0.2, k_fold=5,regularization_lambda=0):
    if hidden_size is None:
        hidden_size = X.shape[1]

    splitter = KFoldEvaluation(random_state=random_state, test_size=test_size, k_fold=k_fold)
    x_test, y_test, folds = splitter.data_spiting(X, y)

    all_fold_results = []
    for i in range(k_fold):
        fold = folds[i]
        x_train, y_train = fold['X_train_fold'] , fold['y_train_fold']
        x_val, y_val     = fold['X_val_fold']   , fold['y_val_fold']

        elm = ExtremeLearningMachine(features_size=x_train.shape[1], hidden_size=hidden_size,activation_function=activation_function,regularization_lambda=regularization_lambda)
        elm.initialize_random_weights(random_seed=random_state)
        elm.fit(x_train, y_train)

        y_pred = elm.predict(x_val.values)
        metrics = EvaluationMatrix(y_val, y_pred)
        all_fold_results.append(metrics.get_all_metrics())

    return EvaluationMatrix.k_fold_metrics(all_fold_results)

In [5]:
def random_seed_cv_validation(X, y, activation_function, hidden_size, test_size=0.2, k_fold=5,regularization_lambda=0):
    results_list = []
    for i in range(40, 61):
        seed_result = cross_validation(
            X,
            y,
            activation_function,
            hidden_size=hidden_size,
            random_state=i,
            test_size=test_size,
            k_fold=k_fold,
            regularization_lambda=regularization_lambda
        )
        results_list.append(seed_result)

    all_results_df = pd.concat(results_list, ignore_index=True)

    return EvaluationMatrix.random_seed_metrics(all_results_df,activation_function,hidden_size,regularization_lambda)

In [6]:
def grid_search_elm(X, y, activation_function, hidden_size_range, test_size=0.2, k_fold=5):
    master_results = []
    total_runs = len(hidden_size_range)
    current_run = 1

    for h_size in hidden_size_range:
        print(f"[{current_run}/{total_runs}] | Hidden Nodes: {h_size}...")

        config_result = random_seed_cv_validation(
            X,
            y,
            activation_function=activation_function,
            hidden_size=h_size,
            test_size=test_size,
            k_fold=k_fold
        )

        master_results.append(config_result)
        current_run += 1

    final_grid_df = pd.concat(master_results, ignore_index=True)

    return final_grid_df

In [7]:
hidden_node_cv = grid_search_elm(X,y,activation_function=sigmoid,hidden_size_range= range(48,53))

[1/5] | Hidden Nodes: 48...
[2/5] | Hidden Nodes: 49...
[3/5] | Hidden Nodes: 50...
[4/5] | Hidden Nodes: 51...
[5/5] | Hidden Nodes: 52...


In [8]:
# 1. Calculate the weighted score (50% Accuracy + 50% MCC)
hidden_node_cv['Weighted_Score'] = (hidden_node_cv['avg_Accuracy_Seed_Mean'] * 0.5) + (hidden_node_cv['avg_MCC_Seed_Mean'] * 0.5)

# 2. Sort by score and extract the top 5 Hidden_Nodes as a list
best_hidden_nodes_list = hidden_node_cv.sort_values(by='Weighted_Score', ascending=False).head(5)['Hidden_Nodes'].tolist()

In [9]:
finalResults = []
for hiddenNodes in best_hidden_nodes_list:
    for lambdaValue in np.arange(7.00, 10.00, 0.25):
        result = random_seed_cv_validation(X,y,sigmoid,hiddenNodes,0.2,5,lambdaValue)
        finalResults.append(result)
final_grids_df = pd.concat(finalResults, ignore_index=True)
final_grids_df

,Hidden_Nodes,Activation,Lambda Value,avg_Accuracy_Seed_Mean,avg_Accuracy_Seed_Std,avg_Accuracy_Seed_Min,avg_Accuracy_Seed_Max,avg_Precision_Seed_Mean,avg_Precision_Seed_Std,avg_Precision_Seed_Min,...,avg_F1-Score_Seed_Min,avg_F1-Score_Seed_Max,avg_Balanced Accuracy_Seed_Mean,avg_Balanced Accuracy_Seed_Std,avg_Balanced Accuracy_Seed_Min,avg_Balanced Accuracy_Seed_Max,avg_MCC_Seed_Mean,avg_MCC_Seed_Std,avg_MCC_Seed_Min,avg_MCC_Seed_Max
0,50,sigmoid,7.00,0.7093,0.0336,0.6471,0.7765,0.7313,0.0387,0.6527,...,0.6051,0.7562,0.7089,0.0337,0.6462,0.7762,0.4235,0.0682,0.2962,0.5615
1,50,sigmoid,7.25,0.7098,0.0330,0.6471,0.7725,0.7325,0.0381,0.6577,...,0.6051,0.7503,0.7094,0.0331,0.6462,0.7723,0.4248,0.0670,0.2999,0.5547
2,50,sigmoid,7.50,0.7100,0.0327,0.6471,0.7725,0.7325,0.0381,0.6577,...,0.6051,0.7503,0.7096,0.0329,0.6462,0.7723,0.4250,0.0665,0.2999,0.5547
3,50,sigmoid,7.75,0.7089,0.0338,0.6431,0.7725,0.7307,0.0394,0.6524,...,0.5994,0.7503,0.7085,0.0339,0.6422,0.7723,0.4226,0.0686,0.2920,0.5547
4,50,sigmoid,8.00,0.7102,0.0332,0.6431,0.7686,0.7316,0.0386,0.6524,...,0.5994,0.7473,0.7098,0.0333,0.6422,0.7683,0.4253,0.0672,0.2920,0.5453
5,50,sigmoid,8.25,0.7107,0.0332,0.6431,0.7725,0.7325,0.0393,0.6573,...,0.5994,0.7505,0.7104,0.0333,0.6422,0.7722,0.4264,0.0673,0.2920,0.5532
6,50,sigmoid,8.50,0.7109,0.0325,0.6471,0.7725,0.7328,0.0382,0.6573,...,0.6025,0.7505,0.7106,0.0326,0.6460,0.7722,0.4268,0.0659,0.3006,0.5532
7,50,sigmoid,8.75,0.7109,0.0321,0.6471,0.7725,0.7332,0.0379,0.6573,...,0.6025,0.7505,0.7106,0.0322,0.6460,0.7722,0.4268,0.0651,0.3006,0.5532
8,50,sigmoid,9.00,0.7113,0.0316,0.6471,0.7725,0.7338,0.0374,0.6573,...,0.6025,0.7505,0.7109,0.0317,0.6460,0.7722,0.4276,0.0641,0.3006,0.5532
9,50,sigmoid,9.25,0.7109,0.0317,0.6471,0.7725,0.7335,0.0376,0.6573,...,0.6025,0.7505,0.7106,0.0318,0.6460,0.7722,0.4269,0.0643,0.3006,0.5532


In [10]:
# Extract the absolute best configuration from the granular Lambda Grid Search
final_grids_df['Weighted_Score'] = (final_grids_df['avg_Accuracy_Seed_Mean'] * 0.5) + (final_grids_df['avg_MCC_Seed_Mean'] * 0.5)
optimal_config = final_grids_df.sort_values(by='Weighted_Score', ascending=False).iloc[0]

opt_hidden = int(optimal_config['Hidden_Nodes'])
opt_lambda = float(optimal_config['Lambda Value'])

print(f"Optimal Configuration Detected -> Hidden Nodes: {opt_hidden} | Lambda: {opt_lambda}")

# Execute K-Fold split to obtain distinct train/val data for benchmarking
from ABC_ELM import ABC_ELM

splitter = KFoldEvaluation(random_state=42, test_size=0.2, k_fold=2)
x_test, y_test, folds = splitter.data_spiting(X, y)

# We will demonstrate the metaheuristic optimization on Fold 0
fold_0 = folds[0]
x_train_f0 = fold_0['X_train_fold']
y_train_f0 = fold_0['y_train_fold']
x_val_f0   = fold_0['X_val_fold']
y_val_f0   = fold_0['y_val_fold']

print("\n[+] Initializing ABC(II)-ELM Optimization...")
# Parameters SN=10, limit=10, iter_max=100 are derived directly from the literature
abc_model = ABC_ELM(
    feature_size=x_train_f0.shape[1],
    hidden_size=opt_hidden,
    activation_function=sigmoid,
    regularization_lambda=opt_lambda,
    SN=10, limit=15, iter_max=1000
)

# Fit the model (this will take longer due to the swarm search space)
abc_model.fit(x_train_f0, y_train_f0)

# Evaluate
y_pred_abc = abc_model.predict(x_val_f0)
eval_metrics = EvaluationMatrix(y_val_f0, y_pred_abc)

print("\n=== ABC-ELM Optimization Results (Fold 0) ===")
for metric, value in eval_metrics.get_all_metrics().items():
    print(f"{metric}: {value:.4f}")

Optimal Configuration Detected -> Hidden Nodes: 51 | Lambda: 8.75

[+] Initializing ABC(II)-ELM Optimization...
1  start
1  end
2  start
2  end
3  start
3  end
4  start
4  end
5  start
5  end
6  start
6  end
7  start
7  end
8  start
8  end
9  start
9  end
10  start
10  end
11  start
11  end
12  start
12  end
13  start
13  end
14  start
14  end
15  start
15  end
16  start
16  end
17  start
17  end
18  start
18  end
19  start
19  end
20  start
20  end
21  start
21  end
22  start
22  end
23  start
23  end
24  start
24  end
25  start
25  end
26  start
26  end
27  start
27  end
28  start
28  end
29  start
29  end
30  start
30  end
31  start
31  end
32  start
32  end
33  start
33  end
34  start
34  end
35  start
35  end
36  start
36  end
37  start
37  end
38  start
38  end
39  start
39  end
40  start
40  end
41  start
41  end
42  start
42  end
43  start
43  end
44  start
44  end
45  start
45  end
46  start
46  end
47  start
47  end
48  start
48  end
49  start
49  end
50  start
50  end
51  st